In [1]:
from transformers import MarianMTModel, MarianTokenizer

class TranslationWithMarianMT:
    def __init__(self):
        # Inizializza i modelli e i tokenizer
        self.models = {
            'en-it': MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-it'),
            'it-en': MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-it-en')
        }
        self.tokenizers = {
            'en-it': MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-it'),
            'it-en': MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-it-en')
        }

    def translate(self, text, source_lang='en', target_lang='it'):
        if source_lang == 'en' and target_lang == 'it':
            model_name = 'en-it'
        elif source_lang == 'it' and target_lang == 'en':
            model_name = 'it-en'
        else:
            raise ValueError(f"Translation from {source_lang} to {target_lang} is not supported.")

        model = self.models[model_name]
        tokenizer = self.tokenizers[model_name]

        # Preparazione input e generazione traduzione
        inputs = tokenizer(text, return_tensors="pt", padding=True)
        translated = model.generate(**inputs)
        
        return tokenizer.decode(translated[0], skip_special_tokens=True)

# Inizializzazione ed esecuzione
translator = TranslationWithMarianMT()

print("TRANSLATION TASK (English to Italian):")
text_en_to_it = "Hello, how are you?"
translated_text_en_to_it = translator.translate(text_en_to_it, source_lang='en', target_lang='it')
print(f"Original: {text_en_to_it}")
print(f"Translated (English to Italian): {translated_text_en_to_it}")

print("\nTRANSLATION TASK (Italian to English):")
text_it_to_en = "Ciao, come stai?"
translated_text_it_to_en = translator.translate(text_it_to_en, source_lang='it', target_lang='en')
print(f"Original: {text_it_to_en}")
print(f"Translated (Italian to English): {translated_text_it_to_en}")

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

c:\Users\A1618apulia\OneDrive - ITS Apulia Digital Maker\Desktop\GENERATIVEAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\A1618apulia\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  343MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

... (16 outputs left out)

source.spm:   0%|          | 0.00/814k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.37M [00:00<?, ?B/s]

TRANSLATION TASK (English to Italian):
Original: Hello, how are you?
Translated (English to Italian): Ciao, come stai?

TRANSLATION TASK (Italian to English):
Original: Ciao, come stai?
Translated (Italian to English): Hi, how are you?


In [4]:
# Importiamo da transformers le classi 'AutoTokenizer' e 'AutoModelForSeq2SeqLM'.
# AutoTokenizer: converte il testo in vettori numerici compresi dal modello e viceversa.
# AutoModelForSeq2SeqLM: carica il modello neurale Sequence-to-Sequence per la traduzione.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Definiamo la classe che gestirà l'intero processo di traduzione
class TranslationWithMarianMT:
    def __init__(self):
        # Creiamo un dizionario che associa le coppie di lingue ai rispettivi modelli su Hugging Face
        self.model_names = {
            'en-it': 'Helsinki-NLP/opus-mt-en-it',  # Modello per tradurre da Inglese a Italiano
            'it-en': 'Helsinki-NLP/opus-mt-it-en'   # Modello per tradurre da Italiano a Inglese
        }
        
        # Inizializziamo due dizionari vuoti che conterranno i modelli e i tokenizer una volta caricati
        self.models = {}
        self.tokenizers = {}

    def _load_model_and_tokenizer(self, pair_key):
        """Metodo privato per caricare in memoria il modello e il tokenizer solo quando servono."""
        # Controlliamo se la coppia di lingue non è ancora stata caricata nei nostri dizionari
        if pair_key not in self.models:
            # Recuperiamo il nome del modello su Hugging Face corrispondente alla chiave (es. 'en-it')
            model_name = self.model_names[pair_key]
            
            # Scarichiamo/carichiamo il tokenizer specifico per questo modello
            self.tokenizers[pair_key] = AutoTokenizer.from_pretrained(model_name)
            
            # Scarichiamo/carichiamo il modello di rete neurale di traduzione
            self.models[pair_key] = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def translate(self, text, source_lang='en', target_lang='it'):
        """Metodo principale che effettua la traduzione del testo fornito."""
        # Componiamo la chiave identificativa della combinazione di lingue (es. 'en-it')
        pair_key = f"{source_lang}-{target_lang}"
        
        # Verifichiamo se la combinazione richiesta è tra quelle supportate dal nostro dizionario
        if pair_key not in self.model_names:
            raise ValueError(f"La traduzione da '{source_lang}' a '{target_lang}' non è supportata.")

        # Carichiamo il modello e il tokenizer necessari (se non sono già in memoria)
        self._load_model_and_tokenizer(pair_key)

        # Estraiamo dalle variabili di istanza il modello e il tokenizer pronti all'uso
        model = self.models[pair_key]
        tokenizer = self.tokenizers[pair_key]

        # FASE 1: TOKENIZZAZIONE
        # Prendiamo la stringa di testo e la trasformiamo in ID numerici (tokens).
        # return_tensors="pt" -> restituisce tensori PyTorch (formato richiesto dal modello).
        # padding=True -> aggiunge automaticamente il riempimento se il testo è lungo o strutturato a blocchi.
        inputs = tokenizer(text, return_tensors="pt", padding=True)
        
        # FASE 2: GENERAZIONE
        # Passiamo i token numerici d'ingresso al modello neurale.
        # Il modello elabora l'input e genera una sequenza di token numerici nella lingua target.
        translated_tokens = model.generate(**inputs)
        
        # FASE 3: DECODIFICA
        # Convertiamo i token numerici generati dal modello di nuovo in una stringa di testo leggibile.
        # translated_tokens[0] -> prende la prima (e unica) frase tradotta generata.
        # skip_special_tokens=True -> rimuove caratteri tecnici di servizio usati dal modello (come <pad>, </s>).
        translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
        
        # Restituiamo il testo tradotto finale
        return translated_text


# --- BLOCCO DI ESECUZIONE INTERATTIVO ---
# Verifica se lo script viene eseguito direttamente (e non importato come modulo)
if __name__ == "__main__":
    
    # Intestazione grafica per l'interfaccia a riga di comando
    print("===========================================")
    print("    TRADUTTORE AUTOMATICO CON MARIANMT     ")
    print("===========================================\n")

    # Creiamo un'istanza della nostra classe traduttore
    translator = TranslationWithMarianMT()

    # RACCOLTA DELLE VARIABILI IN INPUT DALL'UTENTE (Punto 1 delle slide)[cite: 1]
    
    # Chiediamo all'utente di inserire la frase da tradurre.
    # .strip() rimuove eventuali spazi vuoti accidentali all'inizio e alla fine del testo.
    text_to_translate = input("1. Inserisci il testo da tradurre: ").strip()
    
    # Chiediamo la lingua di partenza (es. 'en' o 'it') e la convertiamo in minuscolo con .lower()
    source_language = input("2. Lingua di partenza (es. 'en' o 'it'): ").strip().lower()
    
    # Chiediamo la lingua di destinazione (es. 'it' o 'en') e la convertiamo in minuscolo
    target_language = input("3. Lingua di destinazione (es. 'it' o 'en'): ").strip().lower()

    # Blocco try-except per gestire in modo sicuro eventuali errori (es. lingua non supportata)
    try:
        # Chiamiamo la funzione di traduzione passando le variabili lette in input dall'utente
        result = translator.translate(
            text=text_to_translate, 
            source_lang=source_language, 
            target_lang=target_language
        )
        
        # Stampiamo a schermo il risultato finale in modo chiaro
        print("\n-------------------------------------------")
        print(f"Testo originale [{source_language}]: {text_to_translate}")
        print(f"Traduzione [{target_language}]: {result}")
        print("-------------------------------------------")

    except Exception as error:
        # Se si verifica un errore durante l'esecuzione, lo mostriamo all'utente in modo leggibile
        print(f"\n[ERRORE]: Impossibile completare la traduzione. Dettaglio: {error}")

    TRADUTTORE AUTOMATICO CON MARIANMT     


[ERRORE]: Impossibile completare la traduzione. Dettaglio: La traduzione da 'es' a 'it' non è supportata.
